# NumCompute-Stream — Streaming ML Demo

This notebook demonstrates the **NumCompute-Stream** framework: a NumPy-only
(no scikit-learn, no pandas) toolkit for **streaming / incremental** tree-based
classification.

Through the code segments I am trying to:
1. Load a CSV stream with the custom `io` layer.
2. Build a `Pipeline` of streaming transformers + a model.
3. Train **incrementally** chunk-by-chunk with `partial_fit`, scoring each
   chunk *before* training on it (prequential evaluation).
4. Compare a single Hoeffding tree against an online Random Forest.
5. Visualise the logged metrics with the `visualise` module.


In [ ]:
import sys, os
# the package lives one directory up from demo/; make it importable
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from numcompute_stream import (
    load_csv, iter_chunks,
    Pipeline, Imputer, StandardScaler,
    DecisionTreeClassifier, RandomForestClassifier,
    StreamTrainer, visualise,
)

np.set_printoptions(precision=3, suppress=True)

## 1. Load the stream

`load_csv` is implemented purely on top of the stdlib `csv` module + NumPy.
Missing cells are returned as `NaN`

In [ ]:
DATA_PATH = '../data/stream_data.csv'   # Here we are pointing to out dataset
CHUNK_SIZE = 300

X, y, meta = load_csv(DATA_PATH, target=-1, has_header=True,
                      categorical_target=True)
classes = np.unique(y)
print('X shape :', X.shape)
print('classes :', meta['classes'])
print('missing cells:', int(np.isnan(X).sum()))

## 2. Building a streaming pipeline

Every step supports `partial_fit`. The `Imputer` learns running column means,
the `StandardScaler` keeps Welford running mean/variance, and the model grows
online. The pipeline transforms forward and trains the final estimator in one
`partial_fit` call.

In [ ]:
def make_pipeline(model):
    return Pipeline([
        ('impute', Imputer('mean')),
        ('scale', StandardScaler()),
        ('model', model),
    ])

tree_params = dict(max_depth=10, min_samples_split=60, delta=0.05)
models = {
    'Single tree': make_pipeline(DecisionTreeClassifier(**tree_params)),
    'Random forest': make_pipeline(
        RandomForestClassifier(n_estimators=12, max_features='sqrt',
                               tree_params=tree_params, random_state=0)),
}